In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="hf_pipeline_mrpc_threshold_070",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [3]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=str(device),
    top_k=None,
    function_to_apply="softmax",
)

id2label = model.config.id2label
positive_label = id2label[1]
print(model_name)
print(id2label)
print("positive_label:", positive_label)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

textattack/distilbert-base-uncased-MRPC
{0: 'LABEL_0', 1: 'LABEL_1'}
positive_label: LABEL_1


In [4]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
pair_inputs = [{"text": s1, "text_pair": s2} for s1, s2 in zip(sent1, sent2)]
threshold = 0.70

print("num_pair_inputs:", len(pair_inputs))
print("threshold:", threshold)
print(pair_inputs[0])


num_pair_inputs: 408
threshold: 0.7
{'text': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'text_pair': '" The foodservice pie business does not fit our long-term growth strategy .'}


In [6]:
batch_size = 64
all_outputs = []

for i in tqdm(range(0, len(pair_inputs), batch_size)):
    batch_inputs = pair_inputs[i:i + batch_size]
    batch_outputs = clf(
        batch_inputs,
        batch_size=batch_size,
        truncation=True,
        padding=True,
        max_length=128,
    )
    all_outputs.extend(batch_outputs)

paraphrase_probs = []
for output in all_outputs:
    score_map = {item["label"]: float(item["score"]) for item in output}
    paraphrase_probs.append(score_map[positive_label])

paraphrase_probs = np.array(paraphrase_probs, dtype=np.float32)
y_pred = (paraphrase_probs >= threshold).astype(np.int64)

print("done")


  0%|          | 0/7 [00:00<?, ?it/s]

done


In [ ]:

vault.create_record_list("pipeline_distilbert_threshold", column_names=["prediction", "paraphrase_probs"])

for i in range(len(y_pred)):
    vault.append_record("pipeline_distilbert_threshold", 
                        {
                            "prediction": y_pred[i],
                            "paraphrase_probs": paraphrase_probs[i] ,
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT pipeline_distilbert_threshold"
embedding = get_embeddings(description)
vault.create_description("pipeline_distilbert_threshold", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("pipeline_distilbert_threshold", cat, embedding, prop)

In [7]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], digits=4)
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"], digits=4))


{'accuracy': 0.8480392156862745, 'f1': 0.8896797153024911}
                precision    recall  f1-score   support

not_paraphrase     0.7680    0.7442    0.7559       129
    paraphrase     0.8834    0.8961    0.8897       279

      accuracy                         0.8480       408
     macro avg     0.8257    0.8201    0.8228       408
  weighted avg     0.8469    0.8480    0.8474       408



In [8]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "paraphrase_prob:", float(paraphrase_probs[i]), "label:", id2label[int(y_pred[i])])


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 paraphrase_prob: 0.983514666557312 label: LABEL_1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 paraphrase_prob: 0.18237046897411346 label: LABEL_0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 paraphrase_prob: 0.2533007562160492 label: LABEL_0
sentence1: The AFL-CIO is waiting until October to decide if it will endors

In [9]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "paraphrase_prob:", float(paraphrase_probs[i]))


num_errors: 62
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1 paraphrase_prob: 0.929533839225769
idx: 26
sentence1: Cooley said he expects Muhammad will similarly be called as a witness at a pretrial hearing for Malvo .
sentence2: Lee Boyd Malvo will be called as a witness Wednesday in a pretrial hearing for fellow sniper suspect John Allen Muhammad .
true: 0 pred: 1 paraphrase_prob: 0.8425254225730896
idx: 32
sentence1: Mr Annan also warned the US should not use the war on terror as an excuse to suppress " long-cherished freedoms " .
sentence2: Annan warned that the dangers of extremism after September 11 should not be used as an excuse to suppress " long-cherished " freedoms .
true: 1 pred: 0 paraphrase_prob: 0.6532061696052551
idx: 35
sentence1: Bush wanted 

In [10]:

vault.create_record_list("hf_pipeline_mrpc_threshold_070_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("hf_pipeline_mrpc_threshold_070_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "pipeline_distilbert_threshold": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_threshold_070_summary"
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_mrpc_threshold_070_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_mrpc_threshold_070_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'textattack/distilbert-base-uncased-MRPC',
 'device': 'mps',
 'threshold': 0.7,
 'num_examples': 408,
 'accuracy': 0.8480392156862745,
 'f1': 0.8896797153024911}

In [ ]:
description = "INSERT TEXT HERE ABOUT hf_pipeline_mrpc_threshold_070" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_mrpc_threshold_070", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_mrpc_threshold_070", cat, embedding, prop)